# Uşak Elektrik Firması Arıza Dağıtımı - Karınca Kolonisi Optimizasyonu

**Ad:** Hatukay Duran Alabay  
**Öğrenci No:** 2112721062  
**Senaryo:** 2 (Uşak Elektrik Arıza)  
**GitHub:** [Proje Linki](https://github.com/Hatukay/UsakYolOptimizasyon/)

---

## Proje Açıklaması

Bu proje, Uşak ilindeki elektrik firması arıza giderme rotasını optimize etmek için Karınca Kolonisi Algoritması (Ant Colony Optimization - ACO) kullanmaktadır. Senaryo 2'ye göre, merkezden başlayarak 15 farklı mahalledeki arıza noktalarına en kısa rotayı bulmak amaçlanmaktadır.


In [ ]:
# Gerekli kütüphaneleri import ediyoruz
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
import os
import sys

# Proje klasörlerini Python path'ine ekliyoruz
# Bu sayede core ve data klasörlerindeki modülleri import edebiliriz
# Notebook çalıştırıldığında mevcut dizini path'e ekliyoruz
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

# Kendi yazdığımız modülleri import ediyoruz
from core.ant_algorithm import KarincaKolonisiAlgoritmasi
from core.matrix_utils import mesafe_matrisi_olustur, okleid_mesafesi_hesapla
from data.coordinates import usak_mahalleleri_getir, koordinat_listesi_getir, mahalle_isimleri_getir

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Tüm kütüphaneler başarıyla yüklendi!")


## 1. Veri Seti Hazırlama (Uşak Senaryosu)

Uşak ilindeki 15 farklı mahalle/lokasyon için koordinatları hazırlıyoruz. Merkez noktası başlangıç noktası olacak şekilde ayarlanmıştır.


In [ ]:
# Uşak mahallelerinin koordinatlarını alıyoruz
mahalleler = usak_mahalleleri_getir()
koordinatlar = koordinat_listesi_getir()
mahalle_isimleri = mahalle_isimleri_getir()

# Veriyi DataFrame olarak gösteriyoruz
mahalle_df = pd.DataFrame(mahalleler)
print(f"Toplam {len(mahalleler)} mahalle bulundu.")
print(f"\nBaşlangıç Noktası: {mahalle_isimleri[0]} (Merkez)")
print("\nMahalle Listesi:")
print(mahalle_df.to_string(index=False))


## 2. Mesafe Matrisi Oluşturma

İki nokta arasındaki mesafeleri hesaplamak için mesafe matrisi oluşturuyoruz. 

**Not:** Google Maps API kullanımı için `.env` dosyasında `GOOGLE_MAPS_API_KEY` tanımlanmalıdır. API key yoksa veya hata olursa, Haversine formülü ile matematiksel mesafe hesaplaması yapılacaktır.


In [ ]:
# Google Maps API key'i kontrol ediyoruz
# Güvenlik için .env dosyasından okunması önerilir
# Örnek: os.getenv('GOOGLE_MAPS_API_KEY')
api_key = None  # API key yoksa None bırakıyoruz

# Mesafe matrisini oluşturuyoruz
# matrix_utils.py dosyamızdaki fonksiyonu kullanıyoruz
print("Mesafe matrisi hesaplanıyor...")
mesafe_matrisi, gercek_mesafe_kullanildi = mesafe_matrisi_olustur(koordinatlar, api_key)

if gercek_mesafe_kullanildi:
    print("✓ Google Maps API ile gerçek mesafeler kullanıldı")
else:
    print("⚠ Google Maps API kullanılamadı. Haversine formülü ile mesafe hesaplandı (Mock Mode)")

# Mesafe matrisinin boyutunu kontrol ediyoruz
print(f"\nMesafe matrisi boyutu: {mesafe_matrisi.shape}")
print(f"İlk 5x5 matris örneği:\n{mesafe_matrisi[:5, :5]}")


## 3. Karınca Kolonisi Algoritmasını Çalıştırma

Algoritma parametrelerini ayarlayıp en kısa rotayı buluyoruz:

- **Karınca Sayısı:** 20
- **İterasyon Sayısı:** 50
- **Buharlaşma Oranı:** 0.95
- **Alpha (Feromon Önemi):** 1
- **Beta (Mesafe Önemi):** 2


In [ ]:
# Algoritma parametrelerini ayarlıyoruz
KARINCA_SAYISI = 20
ITERASYON_SAYISI = 50
BUHARLASMA_ORANI = 0.95
ALPHA = 1  # Feromon önemi
BETA = 2   # Mesafe önemi

# Karınca Kolonisi Algoritması nesnesini oluşturuyoruz
# ant_algorithm.py dosyamızdaki KarincaKolonisiAlgoritmasi sınıfını kullanıyoruz
aco = KarincaKolonisiAlgoritmasi(
    mesafe_matrisi=mesafe_matrisi,
    karinca_sayisi=KARINCA_SAYISI,
    alpha=ALPHA,
    beta=BETA,
    buharlasma_orani=BUHARLASMA_ORANI,
    iterasyon_sayisi=ITERASYON_SAYISI
)

# Algoritmayı çalıştırıyoruz
print("=" * 60)
en_iyi_rota, en_iyi_mesafe, iterasyon_gecmisi = aco.calistir()
print("=" * 60)


## 4. Sonuçları Görüntüleme

Bulunan en kısa rotayı ve mesafeyi ekrana yazdırıyoruz.


In [ ]:
# En iyi rotayı mahalle isimleriyle gösteriyoruz
rota_mahalleler = [mahalle_isimleri[i] for i in en_iyi_rota]

print("=" * 60)
print("OPTİMİZE EDİLMİŞ ROTA SONUÇLARI")
print("=" * 60)
print(f"\n📍 En Kısa Mesafe: {en_iyi_mesafe:.2f} km")
print(f"\n🛣️  Rota Sırası ({len(en_iyi_rota)-1} durak):")
print("-" * 60)

for idx, mahalle in enumerate(rota_mahalleler, 1):
    if idx == len(rota_mahalleler):
        print(f"{idx}. {mahalle} (Başlangıç noktasına dönüş)")
    else:
        print(f"{idx}. {mahalle}")

print("-" * 60)
print(f"\nRota Özeti: {' → '.join(rota_mahalleler)}")
print("=" * 60)


## 5. Görselleştirme

### 5.1. İterasyonlara Göre Mesafe Değişim Grafiği (Convergence Graph)

Algoritmanın her iterasyonda bulduğu en iyi mesafenin nasıl iyileştiğini gösteren grafik.


In [ ]:
# Matplotlib ile convergence graph çiziyoruz
plt.figure(figsize=(12, 6))
plt.plot(
    range(1, len(iterasyon_gecmisi) + 1),
    iterasyon_gecmisi,
    marker='o',
    markersize=4,
    linewidth=2,
    color='#1f77b4',
    label='En İyi Mesafe'
)

plt.xlabel('İterasyon', fontsize=12, fontweight='bold')
plt.ylabel('Mesafe (km)', fontsize=12, fontweight='bold')
plt.title('Karınca Kolonisi Algoritması - İterasyonlara Göre Mesafe Değişimi', 
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(fontsize=11)

# En iyi noktayı vurguluyoruz
min_idx = np.argmin(iterasyon_gecmisi)
plt.scatter(min_idx + 1, iterasyon_gecmisi[min_idx], 
           color='red', s=100, zorder=5, label=f'En İyi: {iterasyon_gecmisi[min_idx]:.2f} km')
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()

# İstatistikleri yazdırıyoruz
print(f"\n📊 İstatistikler:")
print(f"   Başlangıç Mesafesi: {iterasyon_gecmisi[0]:.2f} km")
print(f"   En İyi Mesafe: {iterasyon_gecmisi[-1]:.2f} km")
print(f"   İyileştirme: {((iterasyon_gecmisi[0] - iterasyon_gecmisi[-1]) / iterasyon_gecmisi[0] * 100):.2f}%")


### 5.2. Folium ile Harita Görselleştirmesi

Uşak haritası üzerinde mahalleleri işaretleyip bulunan en kısa rotayı çizgi ile birleştiriyoruz.


In [ ]:
# Uşak merkez koordinatları (haritanın merkezi için)
usak_merkez_lat = koordinatlar[0][0]
usak_merkez_lng = koordinatlar[0][1]

# Folium haritası oluşturuyoruz
harita = folium.Map(
    location=[usak_merkez_lat, usak_merkez_lng],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Rota çizgisi için koordinatları hazırlıyoruz
rota_koordinatlari = []
for nokta_idx in en_iyi_rota:
    lat, lng = koordinatlar[nokta_idx]
    rota_koordinatlari.append([lat, lng])

# Rota çizgisini haritaya ekliyoruz
folium.PolyLine(
    rota_koordinatlari,
    color='blue',
    weight=4,
    opacity=0.7,
    popup=f'Optimize Edilmiş Rota ({en_iyi_mesafe:.2f} km)'
).add_to(harita)

# Her mahalleyi haritada işaretliyoruz
for idx, (lat, lng) in enumerate(koordinatlar):
    # Başlangıç noktası (Merkez) için farklı renk ve ikon
    if idx == 0:
        renk = 'green'
        ikon = 'home'
        popup_metni = f"<b>{mahalle_isimleri[idx]}</b><br>Başlangıç Noktası"
    else:
        renk = 'red'
        ikon = 'info-sign'
        rota_sirasi = en_iyi_rota.index(idx)
        popup_metni = f"<b>{mahalle_isimleri[idx]}</b><br>Rota Sırası: {rota_sirasi}"
    
    folium.Marker(
        location=[lat, lng],
        popup=popup_metni,
        icon=folium.Icon(color=renk, icon=ikon),
        tooltip=f"{idx+1}. {mahalle_isimleri[idx]}"
    ).add_to(harita)

# Haritayı gösteriyoruz
harita


## 6. Sonuç Özeti

Algoritmanın bulduğu en kısa rota ve performans metrikleri.


In [ ]:
# Sonuç özeti DataFrame'i oluşturuyoruz
sonuc_ozeti = pd.DataFrame({
    'Metrik': [
        'En Kısa Mesafe (km)',
        'Ziyaret Edilen Mahalle Sayısı',
        'Kullanılan Karınca Sayısı',
        'İterasyon Sayısı',
        'Buharlaşma Oranı',
        'Alpha (Feromon Önemi)',
        'Beta (Mesafe Önemi)',
        'Başlangıç Mesafesi (km)',
        'İyileştirme (%)'
    ],
    'Değer': [
        f"{en_iyi_mesafe:.2f}",
        len(mahalleler) - 1,  # Merkez hariç
        KARINCA_SAYISI,
        ITERASYON_SAYISI,
        BUHARLASMA_ORANI,
        ALPHA,
        BETA,
        f"{iterasyon_gecmisi[0]:.2f}",
        f"{((iterasyon_gecmisi[0] - iterasyon_gecmisi[-1]) / iterasyon_gecmisi[0] * 100):.2f}"
    ]
})

print("=" * 60)
print("SONUÇ ÖZETİ")
print("=" * 60)
print(sonuc_ozeti.to_string(index=False))
print("=" * 60)

# Rota detaylarını tablo olarak gösteriyoruz
print("\n📋 Rota Detayları:")
rota_detay_df = pd.DataFrame({
    'Sıra': range(1, len(en_iyi_rota)),
    'Mahalle': [mahalle_isimleri[i] for i in en_iyi_rota],
    'Enlem': [koordinatlar[i][0] for i in en_iyi_rota],
    'Boylam': [koordinatlar[i][1] for i in en_iyi_rota]
})
print(rota_detay_df.to_string(index=False))


---

## Notlar ve Güvenlik Uyarıları

- **API Key Güvenliği:** Google Maps API kullanımı için API key'inizi `.env` dosyasında saklamanız önerilir. Bu dosya `.gitignore` içine eklenmelidir.
- **Mock Mode:** API key yoksa veya hata durumunda, Haversine formülü ile matematiksel mesafe hesaplaması yapılmaktadır.
- **Algoritma Parametreleri:** Farklı parametrelerle denemeler yaparak daha iyi sonuçlar elde edilebilir.

---

**Proje Tamamlandı!** ✅
